In [1]:
import control as ctrl
import numpy as np
import matplotlib.pyplot as plt

# Diseño del sistema de control del pédulo invertido

Seguimos con el ejemplo de teoría, pero con un pequeño cambio:

In [2]:
w0=2
A=[[0,1],[-w0**2,0]]
B=[[0],[1]]
C=[1,0]
D=0

pendulo=ctrl.ss(A,B,C,D)
pendulo

StateSpace(
array([[ 0.,  1.],
       [-4.,  0.]]),
array([[0.],
       [1.]]),
array([[1., 0.]]),
array([[0.]]),
states=2, outputs=1, inputs=1)

## Obtención de la ley de control

In [3]:
K=ctrl.acker(pendulo.A,pendulo.B,[-2*w0,-2*w0])
K

array([12.,  8.])

## Diseño del estimador completo

In [4]:
L=ctrl.acker(pendulo.A.T,pendulo.C.T,[-4*w0,-4*w0]).T

In [5]:
L

array([16., 60.])

In [6]:
np.linalg.eigvals(pendulo.A-pendulo.B@K)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 1)

In [ ]:
np.linalg.eigvals(pendulo.A-L@pendulo.C)

## Definimos el sistema compensador ( entrada $y$, salida $u$)

In [ ]:
s=pendulo
Acomp=s.A-s.B@K-L@s.C
Bcomp=L
Ccomp=-K
comp=ctrl.ss(Acomp,Bcomp,Ccomp,0)
comp

In [ ]:
np.linalg.eigvals(comp.A)

Vemos que los polos del compensador son estables!

In [ ]:
 comp.zero()

Verificamos que los polos a lazo cerrado del sistema compensado son los adecuados:

In [ ]:
sys_comp = ctrl.feedback(-s*comp)
sys_comp.pole()

Ahora verificamos que el sistema a lazo cerrado tiene efectivamente los polos donde se pretendía. Igualmente será necesario rediseñar para lograr un controlador que sea correcto.

## Sistema completo.

In [ ]:
sys_completo =  ctrl.connect(ctrl.append(s,comp),[[1,2],[2,1]],[1],[1,2])
t,y = ctrl.initial_response(sys_completo, X0=[1,0,0,0], T=np.linspace(0,3,1001))

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(12,8))
ax[0].plot(t,y[0,:],label="sys completo")
ax[0].set_title('Respuesta al escalón en la referencia')
ax[0].set_xlabel("Tiempo")
ax[0].set_ylabel("Salida")
ax[0].grid()
ax[1].plot(t,y[1,:],label="sys completo")
ax[1].set_title('Respuesta al escalón en la referencia')
ax[1].set_xlabel("Tiempo")
ax[1].set_ylabel("Salida")
ax[1].grid()
fig.tight_layout()

In [ ]:
np.linalg.eigvals(sys_completo.A)

# Ubicación óptima de los autovalores del estimador

In [ ]:
B1=B

In [ ]:
def conjugate_tf(G):
    num = ctrl.tf(G).num[0][0]
    den = ctrl.tf(G).den[0][0]
    nume = [num[i]*((-1)**(len(num)%2+1-i)) for i in range(0, len(num))]
    dene = [den[i]*((-1)**(len(den)%2+1-i)) for i in range(0, len(den))]
    return ctrl.tf(nume,dene)

In [ ]:
aux1 = ctrl.ss(A,B1,C,D)
aux2 = conjugate_tf(aux1)
G_srle = ctrl.tf(aux1)*aux2
ctrl.rlocus(G_srle, xlim=[-8,8], ylim=[-10,10]); #aumento los límites para ver la zona donde quiero ubicar el polo
plt.gcf().set_size_inches(8,6)

In [ ]:
k=12000 # lo busco haciendo clcik sobre las lineas del root locus simétrico

In [ ]:
r,k = ctrl.rlocus(G_srle, kvect=[8000],Plot=False)
r

In [ ]:
rsel = r[np.real(r)<0]
rsel

In [ ]:
L = ctrl.place(pendulo.A.T, pendulo.C.T, rsel).T
L

In [ ]:
comp_srl = ctrl.ss(s.A-s.B@K-L@s.C, L, -K, 0)
comp_srl.pole()

In [ ]:
sys_completo_srl =  ctrl.connect(ctrl.append(s,comp_srl),[[1,2],[2,1]],[1],[1,2])
t_srl, y_srl = ctrl.initial_response(sys_completo_srl, X0=[1,0,0,0], T=np.linspace(0,3,1001))

In [ ]:
L

In [ ]:
K

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(12,8))
ax[0].plot(t,y[0,:],label="sys completo 2do orden dominante")
ax[0].plot(t_srl,y_srl[0,:],label="sys completo SRL")
ax[0].set_title('Respuesta al escalón en la referencia')
ax[0].set_xlabel("Tiempo")
ax[0].set_ylabel("Salida")
ax[0].grid()
ax[1].plot(t,y[1,:],label="sys completo 2do orden dominante")
ax[1].plot(t_srl,y_srl[1,:],label="sys completo SRL")
ax[1].set_title('Respuesta al escalón en la referencia')
ax[1].set_xlabel("Tiempo")
ax[1].set_ylabel("Salida")
ax[1].grid()
fig.tight_layout()